# **Catching Bad Data Before It Reaches Your ML Model**
In this workshop, we will learn how to use pytest to automatically validate data before it is used by a machine learning model. We will use a small sample of the COCO 2017 object-detection dataset. Each annotation contains a category ID and a bounding box that describes where an object appears in an image.

**By the end of this workshop, you will be able to:**
- Understand common data validation checks
- Write automated tests with pytest
- Identify invalid object-detection annotations
- Create your own test to check additional data requirements

**What you will need:**
- A Google account to access Google Colab
- Access to the shared workshop Google Drive folder

## **Step 1: Import Libraries**
Before working with the dataset, we need to import the libraries that will help us inspect and validate the data.

In [ ]:
!pip install pytest -q
import pandas as pd
import numpy as np
import pytest

## **Step 2: Load the Dataset**
We will use a prepared sample of 200 annotations from the COCO 2017 training dataset.

**Accessing the dataset:**
1.  Open the shared link: [Data](https://drive.google.com/drive/folders/1sIZwIjIPRMouxkyl9RND-HoICpHW6v3w?usp=drive_link)
2. Right-click the folder **ML_Workshop** and select **Organize** -> **Add shortcut**  
3. Navigate to **All locations** -> **My Drive** -> click **Add**
4. Run the code cells below to mount your Google Drive and load the dataset
> **Important:** Make sure you are signed into Google Colab with the same Google account where you saved the shortcut












In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Define path to dataset shortcut saved in Drive
file_path = "/content/drive/MyDrive/ML_Workshop/coco_sample.csv"

# Load the CVS file into a DataFrame
df = pd.read_csv(file_path)
print("Dataset loaded successfully!\n")

# Inspect the first 5 rows, dimensions, column names, and data types
display(df.head())
print("\nDataset shape:", df.shape)
print("\nColumn names:")
print(df.columns.tolist())
print("\nData types:")
print(df.dtypes)

Dataset loaded successfully!



,annotation_id,image_id,category_id,x,y,width,height,image_width,image_height
0,1598766,493811,62,378.13,0.00,64.14,27.76,640,426
1,197133,549508,1,43.05,115.16,190.49,364.84,640,480
2,2041502,332877,3,117.70,112.68,14.21,11.11,640,360
3,1953056,87091,64,1.66,412.12,33.10,54.63,640,639
4,102066,506673,62,436.20,297.70,55.22,39.57,640,427



Dataset shape: (200, 9)

Column names:
['annotation_id', 'image_id', 'category_id', 'x', 'y', 'width', 'height', 'image_width', 'image_height']

Data types:
annotation_id      int64
image_id           int64
category_id        int64
x                float64
y                float64
width            float64
height           float64
image_width        int64
image_height       int64
dtype: object


## **Step 3: Understand the Dataset**
Each row represents one object annotation from a COCO image. Our dataset has the following columns:

| Column   | Description |
|----------|-------------|
| annotation_id    | Unique annotation identifier        |
| image_id    | Identifier of the image       |  
| category_id | Object category identifier |
| x | Horizontal coordinate of bounding box top-left corner |
| y | Vertical coordinate of bounding box top-left corner |
| width | Bounding box width |
| height | Bounding box height |
| image_width | Original image width |
| image_height | original image height|

COCO is measured in pixels and stores bounding boxes as: [x, y, width, height]




## **Step 4: Inspect the Data**
Before writing automated tests, we should first understand the condition of our dataset. Data inspection helps us identify potential problems, such as: missing values, duplicate rows, unexpected data types, and invalid numerical values.

Our curated sample should have:
*   200 rows and 9 columns
*   No missing values
No duplicate rows.
*   No infinite values
*   No nonpositive bounding box dimensions
*   No negative coordinates
*   No bounding boxes exceeding image boundaries

In [ ]:
print("Missing values:")
print(df.isnull().sum())
print("\nDuplicate rows:", df.duplicated().sum())

# Check numeric columns for infinite values
print("\nInfinite values:")
print(np.isinf(df.select_dtypes(include="number")).sum().sum())

Missing values:
annotation_id    0
image_id         0
category_id      0
x                0
y                0
width            0
height           0
image_width      0
image_height     0
dtype: int64

Duplicate rows: 0

Infinite values:
0


In [ ]:
print("Bounding box dimensions:")
print("Nonpositive widths:", (df["width"] <= 0).sum())
print("Nonpositive heights:", (df["height"] <= 0).sum())

print("\nNegative coordinates:")
print("Negative x coordinates:", (df["x"] < 0).sum())
print("Negative y coordinates:", (df["y"] < 0).sum())

print("\nBounding boxes exceeding image boundaries:")
outside_image = (
    (df["x"] + df["width"] > df["image_width"]) |
    (df["y"] + df["height"] > df["image_height"]))
print(outside_image.sum())

Bounding box dimensions:
Nonpositive widths: 0
Nonpositive heights: 0

Negative coordinates:
Negative x coordinates: 0
Negative y coordinates: 0

Bounding boxes exceeding image boundaries:
0


Our sample passes these initial checks. This does not prove that every annotation is correct. It only means that no issues were found in the checks we performed.

## **Step 5: Understand the Validation Rules**

Now that we have inspected our dataset, we can define the requirements that every annotation should satisfy.

Rule 1: Valid category IDs
*   Each annotation must use a category ID that exists in the COCO dataset

Rule 2: Positive bounding box dimensions
*   The width and height of each bounding box must be greater than zero

Rule 3: Nonnegative coordinates
* The top-left corner of the bounding box cannot have negative coordinates

Rule 4: Bounding boxes must stay within image boundaries
* The bounding box cannot extend beyond the dimensions of its image




## **Step 6: Introduction to Pytest**
Instead of manually checking every annotation, we can automate these checks using pytest, a Python framework that allows us to write and run tests to verify that our code and data meet specific requirements.

A test generally consists of:
* A function whose name begins with test_
* An assert statement that checks a condition
* A result indicating whether the condition passed or failed

Run the code cell below to see an example.


We will use the %%writefile command to create a Python test file directly from our notebook. In our test file, each function will represent a separate validation rule.

In [ ]:
%%writefile test_data.py

def test_positive_width(df):
    assert (df["width"] > 0).all()

def test_positive_height(df):
    assert (df["height"] > 0).all()

Overwriting test_data.py


Variables defined in notebook cells are not automatically available inside a separate Python test file. Pytest supports extending its functionality through plugins, and it also provides a fixture system that allows us to supply data or resources to test functions. We can combine these two features to make our notebook's DataFrame available to our tests.

In [ ]:
# Custom pytest plugin to pass DataFrame into test_data.py
class DataFramePlugin:
    def __init__(self, dataframe):
        self.dataframe = dataframe

    # A fixture makes df available as an argument to any test function
    @pytest.fixture
    def df(self):
        return self.dataframe

# Run the tests
pytest.main(["-v", "test_data.py"], plugins=[DataFramePlugin(df)])

============================= test session starts ==============================
platform linux -- Python 3.13.18, pytest-8.4.2, pluggy-1.7.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content
plugins: langsmith-0.14.17, typeguard-4.6.0, anyio-4.16.0, platformdirs-4.14.3
collecting ... collected 2 items

test_data.py::test_positive_width PASSED                                 [ 50%]
test_data.py::test_positive_height PASSED                                [100%]

============================== 2 passed in 0.02s ===============================


<ExitCode.OK: 0>

Each test will display its name and whether it passed or failed. In this example, both tests pass because every annotation in our sample has a positive bounding-box width and height.

## **Step 7: Writing the Tests**

The category IDs in COCO are not consecutive numbers, so we will use the official set of valid COCO 2017 category IDs. Complete the missing parts of the tests below.

In [ ]:
%%writefile test_data.py

valid_category_ids = {
    1, 2, 3, 4, 5, 6, 7, 8, 9, 10,
    11, 13, 14, 15, 16, 17, 18, 19, 20,
    21, 22, 23, 24, 25, 27, 28, 31, 32,
    33, 34, 35, 36, 37, 38, 39, 40, 41,
    42, 43, 44, 46, 47, 48, 49, 50, 51,
    52, 53, 54, 55, 56, 57, 58, 59, 60,
    61, 62, 63, 64, 65, 67, 70, 72, 73,
    74, 75, 76, 77, 78, 79, 80, 81, 82,
    84, 85, 86, 87, 88, 89, 90}

# Test 1: Check that all category IDs are valid
def test_valid_category_ids(df):
    assert XXX

# Test 2: Check that all bounding box dimensions are positive
def test_valid_box_dimensions(df):
    assert XXX
    assert XXX

# Test 3: Check that all bounding box coordinates are nonnegative
def test_nonnegative_coordinates(df):
    assert XXX
    assert XXX


# Test 4: Check that all bounding boxes stay within image boundaries
def test_boxes_within_image(df):
    assert XXX
    assert XXX

Overwriting test_data.py


####<font color='red'>**TRY IT**</font> &#x1f9e0;: Complete the missing expressions (XXX) using the validation rules from Step 5.



**Hints:**
* Use .isin() to check whether  values belong to a set.
* Use .all() to check whether every row satisfies a condition.
* Remember that the right edge is x + width.
* Remember that the bottom edge is y + height.








In [ ]:
import sys

# Remove cached test_data module so Python reloads newly overwritten file
if 'test_data' in sys.modules:
    del sys.modules['test_data']

# Run pytest with --cache-clear to ensure all updated tests are collected
pytest.main(["-v", "--cache-clear", "test_data.py"], plugins=[DataFramePlugin(df)])

============================= test session starts ==============================
platform linux -- Python 3.13.18, pytest-8.4.2, pluggy-1.7.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content
plugins: langsmith-0.14.17, typeguard-4.6.0, anyio-4.16.0, platformdirs-4.14.3
collecting ... collected 4 items

test_data.py::test_valid_category_ids PASSED                             [ 25%]
test_data.py::test_valid_box_dimensions PASSED                           [ 50%]
test_data.py::test_nonnegative_coordinates PASSED                        [ 75%]
test_data.py::test_boxes_within_image PASSED                             [100%]

============================== 4 passed in 0.05s ===============================


<ExitCode.OK: 0>

## **Step 8: Introduce Invalid Data**
So far, all of our tests have passed because the original sample satisfies our validation rules. We will intentionally introduce an error into a copy of our dataset and then run pytest again to see if they are caught.


In [ ]:
# Create a copy of the original dataset
df_bad = df.copy()

# Modify a few values
df_bad.loc[0, "width"] = -10
df_bad.loc[1, "height"] = -5

In [ ]:
%%writefile test_data.py

def test_positive_width(df):
    assert (df["width"] > 0).all()

def test_positive_height(df):
    assert (df["height"] > 0).all()

Overwriting test_data.py


In [ ]:
import sys

# Clear cached module so Python loads the 2 tests we just wrote
if 'test_data' in sys.modules:
    del sys.modules['test_data']

# Pass df_bad into DataFramePlugin
# Both tests should fail and catch the error
pytest.main(["-v", "test_data.py"], plugins=[DataFramePlugin(df_bad)])

============================= test session starts ==============================
platform linux -- Python 3.13.18, pytest-8.4.2, pluggy-1.7.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content
plugins: langsmith-0.14.17, typeguard-4.6.0, anyio-4.16.0, platformdirs-4.14.3
collecting ... collected 2 items

test_data.py::test_positive_width FAILED                                 [ 50%]
test_data.py::test_positive_height FAILED                                [100%]

=================================== FAILURES ===================================
_____________________________ test_positive_width ______________________________

df =      annotation_id  image_id  category_id       x       y   width  height  \
0          1598766    493811           62...          640           480  
198          480           640  
199          640           426  

[200 rows x 9 columns]

    def test_positive_width(df):
>       assert (df["width"] > 0).all()
E       assert np.False_
E        +  whe

<ExitCode.TESTS_FAILED: 1>

## **Step 9: Key Takeaways**
In this workshop, we learned how to use pytest to automate data validation for an object-detection dataset.

**What we accomplished:**
* Inspected a sample of COCO
annotations.
* Defined requirements for valid object-detection annotations.
* Created automated tests using pytest.
* Identified intentionally introduced data errors.

**Source:** [Pytest Documentation](https://docs.pytest.org/en/stable/)

**Dataset:** [COCO 2017 Annotations](http://images.cocodataset.org/annotations/stuff_annotations_trainval2017.zip)
